In [1]:
import tensorflow as tf
from tensorflow.keras import layers

# Batch = 1
# Sequence length = 4
# Embedding dimension = 4

encoder_input = tf.constant([
    [
        [0.1, 0.2, 0.3, 0.4],   # The
        [0.5, 0.6, 0.7, 0.8],   # animal
        [0.9, 1.0, 1.1, 1.2],   # crossed
        [1.3, 1.4, 1.5, 1.6]    # street
    ]
], dtype=tf.float32)

print("Encoder input shape:", encoder_input.shape)

Encoder input shape: (1, 4, 4)


In [2]:
class SimpleEncoder(layers.Layer):

    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

    def call(self, x):

        # Self-attention
        attention_output = self.attention(
            query=x,
            key=x,
            value=x
        )

        # Residual + normalization
        x = self.norm1(x + attention_output)

        # Feed-forward network
        ffn_output = self.ffn(x)

        # Residual + normalization
        x = self.norm2(x + ffn_output)

        return x

In [3]:
encoder = SimpleEncoder(
    embed_dim=4,
    num_heads=2,
    ff_dim=16
)

In [4]:
encoder_output = encoder(encoder_input)

print("Encoder output shape:", encoder_output.shape)
print(encoder_output.numpy())

Encoder output shape: (1, 4, 4)
[[[-0.5053153  -0.11420382 -1.019675    1.6391943 ]
  [-0.5168547  -0.11325124 -1.0109528   1.6410587 ]
  [-0.5288651  -0.11225018 -1.0017952   1.6429105 ]
  [-0.54130644 -0.11120284 -0.99222314  1.6447325 ]]]


In [5]:
decoder_input = tf.constant([
    [
        [0.2, 0.3, 0.4, 0.5],   # I
        [0.6, 0.7, 0.8, 0.9]    # see
    ]
], dtype=tf.float32)

print("Decoder input shape:", decoder_input.shape)

Decoder input shape: (1, 2, 4)


In [6]:
decoder_self_attention = layers.MultiHeadAttention(
    num_heads=2,
    key_dim=4
)

decoder_self_output = decoder_self_attention(
    query=decoder_input,
    key=decoder_input,
    value=decoder_input,
    use_causal_mask=True
)

print("Decoder self-attention output:")
print(decoder_self_output.numpy())

Decoder self-attention output:
[[[ 0.3596691   0.15474424  0.3676597  -0.53118074]
  [ 0.5773004   0.2678972   0.56594247 -0.81360805]]]


In [7]:
cross_attention = layers.MultiHeadAttention(
    num_heads=2,
    key_dim=4
)

cross_attention_output = cross_attention(
    query=decoder_input,
    key=encoder_output,
    value=encoder_output
)

print("Cross-attention output shape:",
      cross_attention_output.shape)

print(cross_attention_output.numpy())

Cross-attention output shape: (1, 2, 4)
[[[ 0.01739246 -0.37218168  0.5808809  -0.26817235]
  [ 0.01739246 -0.37218544  0.5808811  -0.2681823 ]]]


In [8]:
class SimpleDecoder(layers.Layer):

    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        # Masked self-attention
        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        # Cross-attention
        self.cross_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        # Feed-forward network
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        self.norm3 = layers.LayerNormalization()

    def call(self, x, encoder_output):

        # 1. Masked self-attention
        self_attention_output = self.self_attention(
            query=x,
            key=x,
            value=x,
            use_causal_mask=True
        )

        x = self.norm1(
            x + self_attention_output
        )

        # 2. Cross-attention
        cross_attention_output = self.cross_attention(
            query=x,
            key=encoder_output,
            value=encoder_output
        )

        x = self.norm2(
            x + cross_attention_output
        )

        # 3. Feed-forward network
        ffn_output = self.ffn(x)

        x = self.norm3(
            x + ffn_output
        )

        return x

In [10]:
decoder = SimpleDecoder(
    embed_dim=4,
    num_heads=2,
    ff_dim=16
)

In [11]:
decoder_output = decoder(
    decoder_input,
    encoder_output
)

print("Decoder output shape:", decoder_output.shape)
print(decoder_output.numpy())

Decoder output shape: (1, 2, 4)
[[[-1.5653305   0.37742352  1.1849346   0.00297225]
  [-1.5646855   0.48476413  1.1443384  -0.06441718]]]


In [15]:
vocab_size = 4
output_layer = layers.Dense(vocab_size)

logits = output_layer(decoder_output)

print("Logits shape:", logits.shape)
print(logits.numpy())

Logits shape: (1, 2, 4)
[[[ 1.4413353  -0.87240875  2.1421218   0.8857664 ]
  [ 1.3025742  -0.99524915  2.130159    0.91809773]]]


In [16]:
probabilities = tf.nn.softmax(logits, axis=-1)

print("Probabilities shape:", probabilities.shape)
print(probabilities.numpy())

Probabilities shape: (1, 2, 4)
[[[0.27115166 0.02681427 0.546462   0.1555721 ]
  [0.24575634 0.02469294 0.5622381  0.16731264]]]


In [17]:
predicted_tokens = tf.argmax(probabilities, axis=-1)

print("Predicted token IDs:")
print(predicted_tokens.numpy())

Predicted token IDs:
[[2 2]]
